In [ ]:
import os, sys, json, hashlib, shutil, numpy as np, subprocess, time
from pathlib import Path

COMP = Path('/kaggle/input/competitions/google-verifiable-quantum-circuit-compilation')
WORK = Path('/kaggle/working')
sys.path.insert(0, str(COMP / 'kickmix'))
sys.path.insert(0, str(COMP / 'verifiers'))

SRC = Path('/kaggle/input/datasets/michaelrustad/vqcc-dfthc-r10-merged-groups/src')
sys.path.insert(0, str(SRC))

bp_members = Path('/kaggle/input/datasets/busyaprime/33-qcc-v8-members')
sub_dir = WORK / 'submission'
sub_dir.mkdir(exist_ok=True)
(sub_dir / 'dfthcsos').mkdir(exist_ok=True)

shutil.copy2(bp_members / 'invsqrt.kmb', sub_dir / 'invsqrt.kmb')
shutil.copy2(bp_members / 'dqi.kmb', sub_dir / 'dqi.kmb')
for p in bp_members.glob('dfthcsos__*.kmb'):
    shutil.copy2(p, sub_dir / 'dfthcsos' / p.name.replace('dfthcsos__', ''))
shutil.copy2(bp_members / 'dfthcsos__submission.npz', sub_dir / 'dfthcsos' / 'submission.npz')

print('Setup complete')


In [ ]:
# Run gauge.py with multiple seeds, keep the best
IN = sub_dir / 'dfthcsos' / 'submission.npz'
best_l1 = float('inf')
best_seed = None
best_file = None

for seed in [0, 1, 2, 3, 4]:
    OUT = WORK / f'gauged_seed{seed}.npz'
    print(f'\n=== Seed {seed} ===')
    t0 = time.time()
    result = subprocess.run(
        [sys.executable, str(SRC / 'gauge.py'), str(IN), str(OUT), '--sweeps', '40'],
        capture_output=True, text=True, env=dict(os.environ, PYTHONPATH=str(SRC))
    )
    print(result.stdout)
    print(f'Time: {time.time()-t0:.0f}s')
    
    if OUT.exists():
        # Parse the l1sq from stdout
        for line in result.stdout.split('\n'):
            if 'one-norm term' in line:
                l1 = float(line.split('->')[1].split(',')[0].strip())
                print(f'  L1: {l1}')
                if l1 < best_l1:
                    best_l1 = l1
                    best_seed = seed
                    best_file = OUT
                    print(f'  *** New best! ***')

print(f'\nBest: seed={best_seed}, L1={best_l1}')
if best_file:
    shutil.copy2(best_file, WORK / 'best_gauged.npz')
    print('Saved best to best_gauged.npz')


In [ ]:
# Apply best gauge and verify
if best_file:
    shutil.copy2(WORK / 'best_gauged.npz', sub_dir / 'dfthcsos' / 'submission.npz')
    from dfthcsos.verify import verify as CV
    r = CV(sub_dir / 'dfthcsos')
    print('Full verify:', r['status'], r['score'])
    
    # Package
    zip_path = WORK / 'submission_multigauge.zip'
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        for p in sorted(sub_dir.rglob('*')):
            if p.is_file():
                z.write(p, str(p.relative_to(sub_dir)))
    print('Wrote', zip_path, zip_path.stat().st_size, 'bytes')
